In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("lahore_flats_post_feature_selection.csv")

In [3]:
df.head()

,bedrooms,baths,floors_in_building,area_sqft,servant_quarters,kitchens,store_rooms,drawing_room,furnishing_score,agePossession,luxury_category,floor_category,price
0,4.0,4.0,9.0,3264.0,1.0,1.0,1.0,1,37,1.0,2.0,2.0,4.55
1,4.0,5.0,10.0,3318.4,1.0,1.0,0.0,1,31,1.0,2.0,2.0,4.50
2,3.0,4.0,8.0,2720.0,1.0,1.0,1.0,1,49,1.0,2.0,2.0,3.55
3,4.0,4.0,8.0,2067.2,1.0,2.0,1.0,1,18,1.0,2.0,2.0,4.45
4,4.0,5.0,10.0,3318.4,1.0,1.0,1.0,1,18,1.0,1.0,2.0,4.65


In [4]:
X = df.drop(columns=['price'])
y = df['price']

In [5]:
df.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'furnishing_score', 'agePossession', 'luxury_category',
       'floor_category', 'price'],
      dtype='object')

In [7]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [8]:
columns_to_encode = ["agePossession", "luxury_category", "floor_category"]

In [9]:
y_transformed = np.log1p(y)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

columns_to_encode = ["agePossession", "luxury_category", "floor_category"]

numeric_cols = [
    "bedrooms", "baths", "floors_in_building", "area_sqft",
    "servant_quarters", "kitchens", "store_rooms",
    "drawing_room", "furnishing_score"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), columns_to_encode),
    ],
    remainder="passthrough"
)


In [12]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])


In [13]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring="r2")


In [14]:
scores.mean()

np.float64(0.7057587406774839)

In [16]:
scores.std()

np.float64(0.032722978111008966)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)


In [17]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedrooms', 'baths',
                                                   'floors_in_building',
                                                   'area_sqft',
                                                   'servant_quarters',
                                                   'kitchens', 'store_rooms',
                                                   'drawing_room',
                                                   'furnishing_score']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['agePossession',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [18]:
y_pred = pipeline.predict(X_test)


In [20]:
y_pred = np.expm1(y_pred)

from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test), y_pred)


25.311167851712064

In [ ]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", SVR(kernel="rbf", C=100, gamma="scale", epsilon=0.1))
])

svm_pipeline.fit(X_train, y_train)
svm_pred = svm_pipeline.predict(X_test)

svm_pred = np.expm1(svm_pred)
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test), svm_pred)

0.6591088035346845

In [23]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
svm_scores = cross_val_score(svm_pipeline, X, y_transformed, cv=kfold, scoring="r2")
svm_scores.mean()


np.float64(0.6101990324503062)